[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-hierarchical_classification/examples/hierarchical_classification/hierarchical_classification.ipynb)

# Classify down a tree, and stop when confidence drops

A flat list of every leaf is a long question, and a wrong leaf is worse than an honest coarser label. At each level FRIDA-Decisions chooses among the children of the nodes still in the beam. Code multiplies the probabilities down the path. If the step into a node is not confident, the label falls back to the parent.

`confidence` here is the confidence of the choice that entered the node, not a probability of the whole path. The path probability is the product, and the fallback uses confidence.

Pattern: [Hierarchical classification](https://docs.typesafe.ai/cookbooks/hierarchical_classification). The tree and the six products are synthetic.


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
TREE = {
    "goods": {
        "electronics": "электроника и гаджеты",
        "home": "товары для дома и кухни",
        "grocery": "продукты и напитки",
    },
    "electronics": {
        "phones": "телефоны",
        "audio": "наушники и колонки",
    },
    "home": {
        "kitchen": "техника для кухни",
        "furniture": "мебель",
    },
    "grocery": {
        "tea": "чай и кофе",
        "dairy": "молочные продукты",
    },
    "phones": {
        "smartphones": "смартфоны с сенсорным экраном",
        "button": "кнопочные телефоны",
    },
    "audio": {
        "headphones": "наушники",
        "speakers": "колонки",
    },
    "kitchen": {
        "kettles": "чайники и термопоты",
        "coffee": "кофеварки и кофемолки",
    },
    "furniture": {
        "chairs": "стулья",
        "tables": "столы",
    },
    "tea": {
        "black": "чёрный чай",
        "green": "зелёный чай",
    },
    "dairy": {
        "milk": "молоко",
        "cheese": "сыр",
    },
}

BEAM = 2
MIN_CONFIDENCE = 0.45


def classify(judge, text: str) -> dict:
    """Beam over choice probabilities. Low confidence retreats to the parent."""
    frontier = [("goods", 1.0)]
    parent = {"goods": None}
    step_confidence: dict[str, float] = {}
    step_chosen: dict[str, bool] = {}
    while True:
        expandable = [(node, prob) for node, prob in frontier if node in TREE]
        if not expandable:
            break
        questions = {
            node: {"type": "choice", "instructions": "К какой категории из списка относится товар?",
                   "criteria": TREE[node]}
            for node, _ in expandable
        }
        answers = judge({"state": text, "questions": questions})["answers"]
        nxt = []
        for node, prob in expandable:
            answer = answers[node]
            for child, child_p in answer["probabilities"].items():
                parent[child] = node
                step_confidence[child] = answer["confidence"]
                step_chosen[child] = child == answer["choice"]
                nxt.append((child, prob * child_p))
        nxt.sort(key=lambda item: -item[1])
        frontier = nxt[:BEAM]
    best = max(frontier, key=lambda item: item[1])[0]
    label = best
    while label != "goods" and (not step_chosen.get(label, False)
                                or step_confidence.get(label, 0.0) < MIN_CONFIDENCE):
        label = parent[label]
    return {"leaf": best, "label": label, "path_p": max(item[1] for item in frontier),
            "confidence": step_confidence.get(best)}


## Exact leaf, or at least an ancestor

`leaf` is the end of the best path before the fallback. `label` is what the code would store. A gold hit counts when the stored label is the gold leaf or one of its ancestors.


In [4]:
def run(judge, rows: list[dict]) -> None:
    parent = {}
    for node, kids in TREE.items():
        for child in kids:
            parent[child] = node

    def chain(node: str) -> set[str]:
        found = {node}
        while node in parent:
            node = parent[node]
            found.add(node)
        return found

    exact = coarse = 0
    for row in rows:
        result = classify(judge, row["text"])
        exact += result["label"] == row["gold"]
        coarse += result["label"] in chain(row["gold"])
        mark = "exact" if result["label"] == row["gold"] else ("coarse" if result["label"] in chain(row["gold"]) else "miss")
        print(f"{mark:6} gold {row['gold']:12} label {result['label']:12} "
              f"leaf {result['leaf']:12} path_p={result['path_p']:.3f} "
              f"leaf_conf={None if result['confidence'] is None else round(result['confidence'], 3)}")
    print(f"\nexact {exact}/{len(rows)}, gold or an ancestor of gold {coarse}/{len(rows)}, "
          f"beam {BEAM}, confidence cutoff {MIN_CONFIDENCE}")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-hierarchical-classification/examples/hierarchical_classification/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


exact  gold smartphones  label smartphones  leaf smartphones  path_p=0.998 leaf_conf=0.999
exact  gold button       label button       leaf button       path_p=0.988 leaf_conf=0.942
exact  gold headphones   label headphones   leaf headphones   path_p=0.941 leaf_conf=0.982
exact  gold kettles      label kettles      leaf kettles      path_p=0.960 leaf_conf=0.999
exact  gold chairs       label chairs       leaf chairs       path_p=0.946 leaf_conf=0.79
exact  gold black        label black        leaf black        path_p=0.956 leaf_conf=1.0
exact  gold speakers     label speakers     leaf speakers     path_p=0.647 leaf_conf=0.667
exact  gold milk         label milk         leaf milk         path_p=0.998 leaf_conf=1.0

exact 8/8, gold or an ancestor of gold 8/8, beam 2, confidence cutoff 0.45
